# Agents and Workflows

**Khóa: Claude with the Anthropic API (Anthropic Skilljar)**

Chương tổng hợp: khi nào dùng **Workflow** (luồng cố định bạn điều khiển) vs **Agent** (Claude tự quyết hành trình), và các mẫu thiết kế phổ biến.


## 1. Workflow vs Agent — Khác biệt cốt lõi

| | **Workflow** | **Agent** |
|---|---|---|
| Điều khiển luồng | **Bạn** (code cứng) | **Claude** (tự quyết) |
| Tính dự đoán | Cao, ổn định | Linh hoạt, khó đoán |
| Chi phí/độ trễ | Thấp hơn | Cao hơn |
| Phù hợp | Tác vụ rõ ràng, lặp lại | Tác vụ mở, nhiều bước, khó định trước |

> 🔑 Nguyên tắc: **Bắt đầu đơn giản nhất.** Phần lớn bài toán giải được bằng 1 lệnh gọi hoặc 1 workflow. Chỉ dùng agent khi thực sự cần model tự khám phá.


## 2. Khi nào nên xây Agent? (4 tiêu chí)

Chỉ xây agent khi **CẢ 4** đều thỏa:
1. **Phức tạp**: nhiều bước, khó mô tả hết trước.
2. **Giá trị**: kết quả đáng để đánh đổi chi phí/độ trễ cao hơn.
3. **Khả thi**: Claude làm được loại việc này.
4. **Sửa được lỗi**: lỗi có thể phát hiện & khắc phục (test, review, rollback).

Thiếu 1 trong 4 → quay về workflow hoặc 1 lệnh gọi đơn giản.


In [ ]:
import anthropic
client = anthropic.Anthropic()
MODEL = "claude-opus-4-8"

def chat(prompt, **kw):
    r = client.messages.create(model=MODEL, max_tokens=1024,
                               messages=[{"role": "user", "content": prompt}], **kw)
    return r.content[0].text

## 3. Mẫu Workflow #1 — Prompt Chaining (Nối chuỗi)

Chia tác vụ thành các bước tuần tự, output bước trước → input bước sau. Đã gặp ở bài Prompt Engineering.


In [ ]:
# Viết blog: dàn ý -> bài viết
dan_y = chat("Lập dàn ý 3 ý cho bài viết ngắn về lợi ích của thiền.")
bai = chat(f"Dựa vào dàn ý, viết đoạn ~100 từ:\n{dan_y}")
print(bai)

## 4. Mẫu Workflow #2 — Routing (Định tuyến)

Phân loại đầu vào trước, rồi **chuyển tới luồng xử lý phù hợp**. Ví dụ: phân loại yêu cầu hỗ trợ → kỹ thuật / thanh toán / chung.


In [ ]:
def route_request(user_msg):
    # Bước 1: phân loại
    category = chat(
        f"Phân loại yêu cầu sau thành đúng MỘT từ: KYTHUAT, THANHTOAN, hoặc CHUNG.\n"
        f"Chỉ trả về 1 từ.\n\nYêu cầu: {user_msg}"
    ).strip().upper()
    print(f"  → Định tuyến: {category}")

    # Bước 2: xử lý theo loại (mỗi loại có system prompt riêng)
    systems = {
        "KYTHUAT": "Bạn là kỹ sư hỗ trợ kỹ thuật, trả lời chi tiết từng bước.",
        "THANHTOAN": "Bạn là nhân viên phòng thanh toán, trả lời về hóa đơn, hoàn tiền.",
        "CHUNG": "Bạn là trợ lý chăm sóc khách hàng thân thiện.",
    }
    sys = systems.get(category, systems["CHUNG"])
    return chat(user_msg, system=sys)

print(route_request("Tôi bị tính phí hai lần cho đơn hàng tháng này, làm sao hoàn tiền?"))

## 5. Mẫu Workflow #3 — Parallelization (Song song)

Chia một việc thành nhiều phần **độc lập**, chạy song song rồi gộp kết quả → nhanh hơn & chất lượng hơn. (Ở đây minh họa tuần tự cho đơn giản; thực tế dùng `asyncio` để chạy đồng thời.)


In [ ]:
def review_code(code_snippet):
    aspects = {
        "Bảo mật": "Tìm lỗ hổng bảo mật trong đoạn code.",
        "Hiệu năng": "Tìm vấn đề về hiệu năng trong đoạn code.",
        "Dễ đọc": "Đánh giá tính dễ đọc và đề xuất cải thiện.",
    }
    results = {}
    for name, instruction in aspects.items():
        results[name] = chat(f"{instruction}\n\nCode:\n{code_snippet}")
    return results

sample = "def f(x):\n    return [i for i in range(x) if x % i == 0]"
for aspect, review in review_code(sample).items():
    print(f"=== {aspect} ===\n{review}\n")

## 6. Agent — Vòng lặp tự quyết (Agentic Loop)

Khi tác vụ mở và nhiều bước, đưa Claude **bộ công cụ** và để nó **tự lặp** đến khi xong. Đây là mẫu đã học ở Tool Use & Agentic Search — chính là "agent".


In [ ]:
# Khung agent tổng quát (dùng lại từ bài Tool Use)
def run_agent(user_message, tools, tool_dispatcher):
    messages = [{"role": "user", "content": user_message}]
    while True:
        res = client.messages.create(model=MODEL, max_tokens=1024,
                                     tools=tools, messages=messages)
        if res.stop_reason == "end_turn":
            return next(b.text for b in res.content if b.type == "text")
        messages.append({"role": "assistant", "content": res.content})
        results = []
        for b in res.content:
            if b.type == "tool_use":
                results.append({"type": "tool_result", "tool_use_id": b.id,
                                "content": tool_dispatcher(b.name, b.input)})
        messages.append({"role": "user", "content": results})

print("Khung run_agent() sẵn sàng — ghép với tools của bạn (xem bài Tool Use).")

## 7. Bản đồ quyết định

```
Tác vụ của bạn?
├─ 1 lệnh gọi giải quyết được?  → Single call (đơn giản nhất)
├─ Nhiều bước nhưng CỐ ĐỊNH?    → Workflow (chaining/routing/parallel)
└─ Mở, nhiều bước, khó định trước & thỏa 4 tiêu chí?  → Agent (agentic loop)
```

> 💡 Luôn ưu tiên giải pháp đơn giản nhất đáp ứng được yêu cầu.


---
## ✅ Tóm tắt

- **Workflow**: bạn điều khiển luồng (cố định) — ổn định, rẻ. Mẫu: **Chaining, Routing, Parallelization**.
- **Agent**: Claude tự quyết hành trình qua vòng lặp tool use — linh hoạt, mạnh, đắt hơn.
- Chỉ xây agent khi thỏa **4 tiêu chí** (phức tạp, giá trị, khả thi, sửa được lỗi).
- **Bắt đầu đơn giản nhất** rồi mới nâng cấp khi cần.
